In [ ]:
# 🔁 Clean install correct versions
!pip uninstall -y transformers
!pip install transformers==4.41.2 datasets==2.19.1 evaluate==0.4.2

# 🔁 Restart the runtime to apply changes
import os
os.kill(os.getpid(), 9)

Found existing installation: transformers 4.53.1
Uninstalling transformers-4.53.1:
  Successfully uninstalled transformers-4.53.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 72.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.2
    Uninstalling tokenizers-0.21.2:
      Successfully uninstalled tokenizers-0.21.2
  Attempting uninstall: datasets
    Found existing installation

In [1]:
import transformers
print(transformers.__version__)


4.53.2


In [ ]:

import torch
from torch.utils.data import DataLoader
from transformers import DistilBertTokenizerFast, DistilBertForQuestionAnswering, AdamW
from datasets import load_dataset
from tqdm import tqdm

# 1. Load small QA dataset
dataset = load_dataset("squad", split='train[:1%]')
dataset = dataset.train_test_split(test_size=0.1)

# 2. Tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def preprocess(example):
    return tokenizer(
        example["question"], example["context"],
        truncation=True,
        padding="max_length",
        max_length=384,
        return_offsets_mapping=True,
        return_tensors="pt"
    )

# 3. Encode dataset
def encode_dataset(dataset):
    input_ids = []
    attention_mask = []
    start_positions = []
    end_positions = []

    for example in tqdm(dataset):
        tokenized = tokenizer(
            example["question"],
            example["context"],
            truncation="only_second",
            max_length=384,
            stride=128,
            return_overflowing_tokens=False,
            return_offsets_mapping=True,
            padding="max_length"
        )

        start_char = example["answers"]["answer_start"][0]
        end_char = start_char + len(example["answers"]["text"][0])

        offsets = tokenized["offset_mapping"]
        context = example["context"]

        start_token = end_token = 0
        for i, (start, end) in enumerate(offsets):
            if start <= start_char < end:
                start_token = i
            if start < end_char <= end:
                end_token = i

        input_ids.append(torch.tensor(tokenized["input_ids"]))
        attention_mask.append(torch.tensor(tokenized["attention_mask"]))
        start_positions.append(torch.tensor(start_token))
        end_positions.append(torch.tensor(end_token))

    return DataLoader(list(zip(input_ids, attention_mask, start_positions, end_positions)), batch_size=4)

train_loader = encode_dataset(dataset["train"])
test_loader = encode_dataset(dataset["test"])

# 4. Model
model = DistilBertForQuestionAnswering.from_pretrained("distilbert-base-uncased")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 5. Optimizer
optimizer = AdamW(model.parameters(), lr=3e-5)

# 6. Training loop
def train_model(epochs=2):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in tqdm(train_loader):
            input_ids, attention_mask, start_pos, end_pos = [b.squeeze().to(device) for b in batch]

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                start_positions=start_pos,
                end_positions=end_pos
            )

            loss = outputs.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

train_model()

# 7. Evaluate on test
def evaluate_model():
    model.eval()
    exact_match = 0
    total = 0

    for batch in tqdm(test_loader):
        input_ids, attention_mask, start_pos, end_pos = [b.squeeze().to(device) for b in batch]

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            start_preds = torch.argmax(outputs.start_logits, dim=1)
            end_preds = torch.argmax(outputs.end_logits, dim=1)

        exact_match += ((start_preds == start_pos) & (end_preds == end_pos)).sum().item()
        total += input_ids.size(0)

    print(f"Exact Match Accuracy: {100 * exact_match / total:.2f}%")

evaluate_model()


Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Epoch 1, Loss: 783.2688


100%|██████████| 197/197 [24:52<00:00,  7.58s/it]


Epoch 2, Loss: 444.1423


100%|██████████| 22/22 [00:51<00:00,  2.34s/it]

Exact Match Accuracy: 19.32%


In [ ]:
def answer_question(question, context):
    model.eval()
    inputs = tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=384,
        return_offsets_mapping=True
    ).to(device)

    offset_mapping = inputs.pop("offset_mapping")
    with torch.no_grad():
        outputs = model(**inputs)
        start_logits = outputs.start_logits
        end_logits = outputs.end_logits

        start_idx = torch.argmax(start_logits)
        end_idx = torch.argmax(end_logits)

        # If end < start, skip
        if end_idx < start_idx:
            print("⚠️ Unable to find valid answer span.")
            return

        offsets = offset_mapping[0]
        start_char = offsets[start_idx][0].item()
        end_char = offsets[end_idx][1].item()

        answer = context[start_char:end_char]

        print(f"\n💬 Question: {question}")
        print(f"📚 Answer: {answer.strip()}")


In [ ]:
#  Your own question-context test
context = """
Hi, My name is Aayush Maharjan. I am currently at Day 46 of my 60 Days of Learning with Leapfrog Journey.
"""

question = "what is my Name?"

answer_question(question, context)



💬 Question: what is my Name?
📚 Answer: Aayush Maharjan


In [ ]:
question = "What Day am I at?"

answer_question(question, context)


💬 Question: What Day am I at?
📚 Answer: 46


In [ ]:
question = "Total days of learning?"

answer_question(question, context)

⚠️ Unable to find valid answer span.
